# Oracle Problem Exercise (SuT: `calculate_shipping_fee`)

The goal of this exercise is to experience the following three approaches firsthand.

1. **Requirements-based manual oracle**: read the business logic and compute expected outputs manually.
2. **Implicit oracle**: express intended true ranges and properties as assertions.
3. **Pseudo-oracle generation**: build an independent implementation (`my_calculate_shipping_fee`) and use it as an oracle.

In [ ]:
def calculate_shipping_fee(
    order_total: int,
    weight_kg: float,
    distance_km: int,
    is_island: bool,
    membership: str,
    coupon_type: str,
) -> int:
    """SuT: Calculate shipping fee based on business rules."""
    fee = 0

    # 1) base fee
    if order_total <= 40000:    # Pkb
        fee += 3000

    # 2) weight surcharge
    if weight_kg < 5:    # Pkb
        fee += 0
    elif weight_kg <= 20:
        fee += 2000
    else:
        fee += 5000     

    # 3) distance surcharge
    if distance_km <= 10:
        fee += 0
    elif distance_km <= 50:
        fee += 1000
    else:
        fee += 3000

    # 4) island surcharge
    if is_island:
        fee += 4000

    # 5) membership discount
    if membership == "WOW":
        fee = fee // 2

    # 6) coupon discount
    if coupon_type == "NEW_USER":
        fee -= 2000

    # 7) lower bound
    return max(fee, 0)


# Smoke check
print(calculate_shipping_fee(30000, 10.0, 20, False, "WOW", "NONE"))

## Chapter 1. Requirements-based Manual Oracle

First, read the SuT business logic and manually compute expected outputs.

Business rules for manual oracle writing:

- If `order_total` is less than 40,000, charge a base shipping fee of 3,000.
- If `weight_kg` is 5 or less, apply no weight surcharge.
- If `weight_kg` is over 5 and up to 20, apply a weight surcharge of 2,000.
- If `weight_kg` is over 20, apply a weight surcharge of 5,000.
- If `distance_km` is 10 or less, apply no distance surcharge.
- If `distance_km` is over 10 and up to 50, apply a distance surcharge of 1,000.
- If `distance_km` is over 50, apply a distance surcharge of 3,000.
- If `is_island` is true, apply an island surcharge of 4,000.
- If membership is `WOW`, halve the fee computed so far.
- If coupon type is `NEW_USER`, subtract 2,000 from the fee.
- The final fee must not be negative; clamp values below 0 to 0.

Now write **test input and expected output pairs manually** based on these rules.

In [ ]:
# Chapter 1: manual oracle = create (input, expected) pairs by hand.
# The expected values below are computed manually from the business rules.

manual_test_cases = [
    (
        {
            "order_total": 10000,
            "weight_kg": 3.0,
            "distance_km": 5,
            "is_island": False,
            "membership": "NONE",
            "coupon_type": "NONE",
        },
        3000,  # expected (manual)
    ),
    (
        {
            "order_total": 45000,
            "weight_kg": 21.0,
            "distance_km": 70,
            "is_island": True,
            "membership": "NONE",
            "coupon_type": "NONE",
        },
        12000,  # expected (manual)
    ),
    (
        {
            "order_total": 39000,
            "weight_kg": 20.0,
            "distance_km": 50,
            "is_island": False,
            "membership": "WOW",
            "coupon_type": "NEW_USER",
        },
        1000,  # expected (manual)
    ),
    (
        {
            "order_total": 50000,
            "weight_kg": 1.0,
            "distance_km": 1,
            "is_island": False,
            "membership": "NONE",
            "coupon_type": "NEW_USER",
        },
        0,  # expected (manual)
    ),
]

for i, (test_input, expected) in enumerate(manual_test_cases, start=1):
    actual = calculate_shipping_fee(**test_input)
    print(f"case#{i}: expected={expected}, actual={actual}, pass={expected == actual}")

### Chapter 1 Reflection

Consider the following questions.

- Which rule was most confusing while creating manual oracles?
- Which boundary values (`40000`, `5`, `20`, `10`, `50`) are most error-prone?
- What are the limitations of writing expected values manually?

> Key takeaway: **Even with a clear spec, consistent expected values are hard to produce.**

## Chapter 2. Implicit Oracle (Assertion-based)

In this chapter, you do not compute exact expected values.
Instead, you write assertions for **properties that must always hold**.

Example properties:

- The result must never be negative.
- For otherwise identical inputs, `is_island=True` should not be cheaper than `False`.
- For otherwise identical inputs, `membership="WOW"` should not be more expensive than `"NONE"`.
- For otherwise identical inputs, `coupon_type="NEW_USER"` should not be more expensive than `"NONE"`.

This approach is useful for finding suspicious behavior even when exact expected values are unavailable.

In [ ]:
# Chapter 2: implicit oracle examples (simple assertion list)
# Instead of exact expected values, verify bound properties derived from business rules.

# 1) Shipping fee must be non-negative in every case.
case_1 = {
    "order_total": 50000,
    "weight_kg": 1.0,
    "distance_km": 1,
    "is_island": False,
    "membership": "NONE",
    "coupon_type": "NEW_USER",
}
fee_1 = calculate_shipping_fee(**case_1)
print("case_1 fee:", fee_1)
assert fee_1 >= 0

# 2) If order_total is below 40,000, fee should be at least the base fee (3,000).
case_2 = {
    "order_total": 30000,
    "weight_kg": 2.0,
    "distance_km": 5,
    "is_island": False,
    "membership": "NONE",
    "coupon_type": "NONE",
}
fee_2 = calculate_shipping_fee(**case_2)
print("case_2 fee:", fee_2)
assert fee_2 >= 3000

# 3) If weight is over 20kg, fee should include at least the 5,000 weight surcharge.
case_3 = {
    "order_total": 50000,
    "weight_kg": 25.0,
    "distance_km": 5,
    "is_island": False,
    "membership": "NONE",
    "coupon_type": "NONE",
}
fee_3 = calculate_shipping_fee(**case_3)
print("case_3 fee:", fee_3)
assert fee_3 >= 5000

# 4) If distance is over 50km and destination is island, fee should be at least 7,000.
case_4 = {
    "order_total": 50000,
    "weight_kg": 1.0,
    "distance_km": 70,
    "is_island": True,
    "membership": "NONE",
    "coupon_type": "NONE",
}
fee_4 = calculate_shipping_fee(**case_4)
print("case_4 fee:", fee_4)
assert fee_4 >= 7000

# 5) If order_total is below 40,000 and weight is over 20kg, fee should be at least 8,000.
case_5 = {
    "order_total": 35000,
    "weight_kg": 21.0,
    "distance_km": 5,
    "is_island": False,
    "membership": "NONE",
    "coupon_type": "NONE",
}
fee_5 = calculate_shipping_fee(**case_5)
print("case_5 fee:", fee_5)
assert fee_5 >= 8000

print("All simple implicit-oracle checks passed.")

### Chapter 2 Reflection

- What kinds of bugs are well-detected by implicit oracles?
- What kinds of bugs are easy to miss if you rely only on implicit oracles?
- What advantages do you get by combining exact and implicit oracles?

> Key takeaway: **You can detect defects through must-hold properties even without exact expected values.**

## Chapter 3. Pseudo-oracle Generation

Now write an independent second implementation, `my_calculate_shipping_fee`, and use it as a pseudo-oracle.

Key principles:

- Do not copy and paste the SuT code.
- Use a different structure or calculation order to increase independence.
- If the two implementations disagree, investigate which one is wrong.

This approach enables automated cross-checking over many inputs.

In [ ]:
def my_calculate_shipping_fee(
    order_total: int,
    weight_kg: float,
    distance_km: int,
    is_island: bool,
    membership: str,
    coupon_type: str,
) -> int:
    # TODO: Implement your own pseudo-oracle function.

    return 0


pseudo_cases = [
    {"order_total": 39999, "weight_kg": 5.0, "distance_km": 10, "is_island": False, "membership": "NONE", "coupon_type": "NONE"},
    {"order_total": 40000, "weight_kg": 5.01, "distance_km": 11, "is_island": False, "membership": "WOW", "coupon_type": "NONE"},
    {"order_total": 10000, "weight_kg": 21.0, "distance_km": 51, "is_island": True, "membership": "WOW", "coupon_type": "NEW_USER"},
    {"order_total": 80000, "weight_kg": 1.0, "distance_km": 1, "is_island": False, "membership": "NONE", "coupon_type": "NEW_USER"},
]

for i, case in enumerate(pseudo_cases, start=1):
    sut = calculate_shipping_fee(**case)
    oracle = my_calculate_shipping_fee(**case)
    print(f"case#{i}: sut={sut}, pseudo_oracle={oracle}, match={sut == oracle}")

### Chapter 3 Reflection

- If pseudo-oracle and SuT results disagree, what evidence helps you narrow down the root cause?
- How can you distinguish a SuT bug from a pseudo-oracle bug?
- What are the strengths and risks of pseudo-oracles (including duplicated mistakes)?

Read this quote again.

> "Programs which were written in order to determine the answer in the
first place. There would be no need
to write such programs, if the correct
answer were known."  
> — *On testing non-testable programs, 1982*

> Key takeaway: **The moment you write another program to decide correctness, the oracle problem reappears.**

In [38]:
# Optional: simple bulk comparison for pseudo-oracle
order_totals = [10000, 39999, 40000, 70000]
weights = [1.0, 5.0, 5.1, 20.0, 20.1]
distances = [1, 10, 11, 50, 51]
islands = [False, True]
memberships = ["NONE", "WOW"]
coupons = ["NONE", "NEW_USER"]

mismatches = []

for ot in order_totals:
    for w in weights:
        for d in distances:
            for i in islands:
                for m in memberships:
                    for c in coupons:
                        case = {
                            "order_total": ot,
                            "weight_kg": w,
                            "distance_km": d,
                            "is_island": i,
                            "membership": m,
                            "coupon_type": c,
                        }
                        sut = calculate_shipping_fee(**case)
                        oracle = my_calculate_shipping_fee(**case)
                        if sut != oracle:
                            mismatches.append((case, sut, oracle))

print(f"total mismatches: {len(mismatches)}")
for idx, (case, sut, oracle) in enumerate(mismatches[:10], start=1):
    print(f"{idx}. case={case}, sut={sut}, pseudo_oracle={oracle}")

if mismatches:
    print("\nTODO: pick one mismatch and decide which implementation is correct using the business rules.")
else:
    print("\nAll sampled cases matched. Try adding more boundary-value cases.")

total mismatches: 755
1. case={'order_total': 10000, 'weight_kg': 1.0, 'distance_km': 1, 'is_island': False, 'membership': 'NONE', 'coupon_type': 'NONE'}, sut=3000, pseudo_oracle=0
2. case={'order_total': 10000, 'weight_kg': 1.0, 'distance_km': 1, 'is_island': False, 'membership': 'NONE', 'coupon_type': 'NEW_USER'}, sut=1000, pseudo_oracle=0
3. case={'order_total': 10000, 'weight_kg': 1.0, 'distance_km': 1, 'is_island': False, 'membership': 'WOW', 'coupon_type': 'NONE'}, sut=1500, pseudo_oracle=0
4. case={'order_total': 10000, 'weight_kg': 1.0, 'distance_km': 1, 'is_island': True, 'membership': 'NONE', 'coupon_type': 'NONE'}, sut=7000, pseudo_oracle=0
5. case={'order_total': 10000, 'weight_kg': 1.0, 'distance_km': 1, 'is_island': True, 'membership': 'NONE', 'coupon_type': 'NEW_USER'}, sut=5000, pseudo_oracle=0
6. case={'order_total': 10000, 'weight_kg': 1.0, 'distance_km': 1, 'is_island': True, 'membership': 'WOW', 'coupon_type': 'NONE'}, sut=3500, pseudo_oracle=0
7. case={'order_total

## ☕ Coffee Break Thinking: Reframing the Oracle Problem

In this exercise, you validated the same SuT in three different ways.

- **Manual oracle**: manually created `(input, expected)` pairs.
- **Implicit oracle**: checked must-hold properties (lower bounds / ranges) instead of exact values.
- **Pseudo-oracle**: compared SuT outputs against an independent implementation.

Pause for a moment and think about these questions.

1. Which assumptions does your oracle depend on?
2. How can you check whether your oracle itself is wrong?
3. When combining manual / implicit / pseudo approaches, how can each method compensate for the others?
4. Which additional boundary-value inputs would improve oracle confidence in the next iteration?

> Takeaway: **Testing is not only about known answers; it is also about building trustworthy evidence when exact answers are unknown.**